In [1]:
import pandas as pd
import numpy as np
import os

from pandas import read_csv

usages = ["directive", "feedback_loop", "learning", "filtered", "task_iteration", "validation"]

In [2]:

claude = read_csv("../claude/202601/aei_raw_claude_ai_2025-11-13_to_2025-11-20.csv")

claude = claude[claude["variable"] == "onet_task_collaboration_count"][["cluster_name", "value"]]
claude[["task_name", "usage"]] = claude["cluster_name"].str.split('::', expand = True)
#claude = claude[~claude["usage"].isin(["not_classified", "none"])]
claude = claude.drop("cluster_name", axis = 1)
claude = claude.pivot(index = "task_name", columns = "usage", values = "value").fillna(0).rename(
    columns = {
        "feedback loop":"feedback_loop",
        "task iteration":"task_iteration"
    }
).assign(
    filtered = lambda x: x["not_classified"] + x["none"],
    total    = lambda x: x[usages].sum(axis = 1)
).drop(["not_classified", "none"], axis = 1).reset_index()

claude = claude[["task_name", "directive", "feedback_loop", "learning", "filtered", "task_iteration", "validation", "total"]]
claude_convs = claude["total"].sum()

print(claude.head())

usage                                          task_name  directive  \
0      accompany buyers during visits to and inspecti...        0.0   
1              act as advisers to student organizations.       33.0   
2      act as an intermediary in negotiations between...       17.0   
3      act as an intermediary in negotiations between...       15.0   
4      act as liaisons between on-site managers or te...        0.0   

usage  feedback_loop  learning  filtered  task_iteration  validation  total  
0                0.0       0.0      21.0             0.0         0.0   21.0  
1                0.0       0.0      17.0            64.0         0.0  114.0  
2                0.0       0.0       9.0            46.0         0.0   72.0  
3                0.0       0.0       9.0            28.0         0.0   52.0  
4                0.0       0.0      29.0             0.0         0.0   29.0  


In [3]:
enterprise = read_csv("../claude/202601/aei_raw_1p_api_2025-11-13_to_2025-11-20.csv")

enterprise = enterprise[enterprise["variable"] == "onet_task_collaboration_count"][["cluster_name", "value"]]
enterprise[["task_name", "usage"]] = enterprise["cluster_name"].str.split('::', expand = True)
#enterprise = enterprise[~enterprise["usage"].isin(["not_classified", "none"])]
enterprise = enterprise.drop("cluster_name", axis = 1)
enterprise = enterprise.pivot(index = "task_name", columns = "usage", values = "value").fillna(0).rename(
    columns = {
        "feedback loop":"feedback_loop",
        "task iteration":"task_iteration"
    }
).assign(
    filtered = lambda x: x["not_classified"] + x["none"],
    total    = lambda x: x[usages].sum(axis = 1)
).drop(["not_classified", "none"], axis = 1).reset_index()

enterprise = enterprise[["task_name", "directive", "feedback_loop", "learning", "filtered", "task_iteration", "validation", "total"]]
enterprise_convs = enterprise["total"].sum()

print(enterprise.head())

usage                                          task_name  directive  \
0      access computerized financial information to a...       33.0   
1      accompany buyers during visits to and inspecti...        0.0   
2      act as a liaison between a ship's captain and ...        0.0   
3      act as an intermediary in negotiations between...       42.0   
4      act as an intermediary in negotiations between...       94.0   

usage  feedback_loop  learning  filtered  task_iteration  validation  total  
0                0.0       0.0       9.0             0.0         0.0   42.0  
1                0.0       0.0      17.0             0.0         0.0   17.0  
2                0.0       0.0      15.0             0.0         0.0   15.0  
3                0.0       0.0      11.0             0.0         0.0   53.0  
4                0.0       0.0       6.0             0.0         0.0  100.0  


In [4]:
november_claude = claude.copy()
november_claude[usages] = november_claude[usages].div(november_claude["total"], axis = 0)
november_claude["pct"]  = november_claude["total"] / claude_convs
november_claude = november_claude.drop("total", axis =1)
print(november_claude)
november_claude.to_csv("../resources/automation_vs_augmentation_by_task_v3_claude.csv", index = False)


usage                                          task_name  directive  \
0      accompany buyers during visits to and inspecti...   0.000000   
1              act as advisers to student organizations.   0.289474   
2      act as an intermediary in negotiations between...   0.236111   
3      act as an intermediary in negotiations between...   0.288462   
4      act as liaisons between on-site managers or te...   0.000000   
...                                                  ...        ...   
3164   write, design, or edit web page content, or di...   0.486352   
3165   write, present, and publish reports that recor...   0.346856   
3166   write, review, or execute plans for testing ne...   0.000000   
3167   write, review, or maintain engineering documen...   0.357143   
3168   write, update, and maintain computer programs ...   0.326557   

usage  feedback_loop  learning  filtered  task_iteration  validation       pct  
0           0.000000  0.000000  1.000000        0.000000    0.0000

In [6]:
november = pd.concat([claude, enterprise], axis = 0, ignore_index= True).fillna(0).copy().sort_values(by='task_name')
november = november.groupby("task_name").sum()
november[usages] = november[usages].div(november["total"], axis = 0)
november["pct"]  = november["total"] / (claude_convs + enterprise_convs)
november = november.drop("total", axis =1).reset_index()
print(november)
november.to_csv("../resources/automation_vs_augmentation_by_task_v3.csv", index = False)


usage                                          task_name  directive  \
0      access computerized financial information to a...   0.785714   
1      accompany buyers during visits to and inspecti...   0.000000   
2      act as a liaison between a ship's captain and ...   0.000000   
3              act as advisers to student organizations.   0.289474   
4      act as an intermediary in negotiations between...   0.472000   
...                                                  ...        ...   
3592   write, design, or edit web page content, or di...   0.653697   
3593   write, present, and publish reports that recor...   0.519011   
3594   write, review, or execute plans for testing ne...   0.000000   
3595   write, review, or maintain engineering documen...   0.418831   
3596   write, update, and maintain computer programs ...   0.463301   

usage  feedback_loop  learning  filtered  task_iteration  validation       pct  
0           0.000000  0.000000  0.214286        0.000000    0.0000